In [6]:
import torch

# 1) load it back
sd = torch.load(
    "./output/Med3DVLM-Qwen-2.5-7B-finetune/model_with_lora.bin",
    map_location="cpu",
)

# 2) print how many keys, plus names & shapes
print(f"Loaded {len(sd)} tensors:")
for name, tensor in sd.items():
    print(f"  {name:60s} — {tuple(tensor.shape)}")


Loaded 1205 tensors:
  base_model.model.model.embed_tokens.weight                   — (0,)
  base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight — (0,)
  base_model.model.model.layers.0.self_attn.q_proj.base_layer.bias — (3584,)
  base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight — (16, 3584)
  base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight — (3584, 16)
  base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight — (0,)
  base_model.model.model.layers.0.self_attn.k_proj.base_layer.bias — (512,)
  base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight — (16, 3584)
  base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight — (512, 16)
  base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight — (0,)
  base_model.model.model.layers.0.self_attn.v_proj.base_layer.bias — (512,)
  base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight — (16, 3584)
  base_model.model.mode

In [7]:
from src.model.llm import VLMQwenForCausalLM
merged = VLMQwenForCausalLM.from_pretrained("./models/Med3DVLM-Qwen-2.5-7B")

keys = merged.state_dict().keys()
print(any("lora_" in k for k in keys))  # should be False

/fs04/scratch2/iq38/biomed_model_pmct/data/minigptmed/MiniGPT-Med/miniconda3/miniconda/conda/envs/med3dvlm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 7/7 [00:01<00:00,  6.98it/s]

False


In [11]:
import torch

# load base and merged
base = VLMQwenForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    cache_dir="/home/sdef0001/iq38_scratch/nmdid/cache/huggingface",
    local_files_only=True,
)

for k in base.state_dict().keys():
    if "layers.0.mlp.up_proj" in k and "weight" in k:
        print(k)

You are using a model of type qwen2 to instantiate a model of type vlm_qwen. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.23it/s]

model.layers.0.mlp.up_proj.weight


In [13]:
merged = VLMQwenForCausalLM.from_pretrained("./models/Med3DVLM-Qwen-2.5-7B")

# key = "model.layers.0.mlp.up_proj.weight"
key = "model.layers.0.mlp.down_proj.weight"
w0  = base.state_dict()[key]
w1  = merged.state_dict()[key]

# these won’t be exactly equal
diff = (w1 - w0).abs().sum().item()
print(f"Total Δ for this proj: {diff:.4f}")

# If diff > 0, your LoRA contribution really got added into the saved weights.

Loading checkpoint shards: 100%|██████████| 7/7 [00:01<00:00,  6.09it/s]


Total Δ for this proj: 867.7760


In [16]:
from tqdm import tqdm

diffs = []
base_sd   = base.state_dict()
merge_sd  = merged.state_dict()

for k, v0 in tqdm(base_sd.items(), total=len(base_sd), desc="Checking params"):
    v1 = merge_sd.get(k)
    if v1 is None or v0.shape != v1.shape:
        continue
    if v0.ndim < 2:
        continue
    d = (v1 - v0).abs().sum().item()
    if d > 1e-6:
        diffs.append((k, d))

Checking params: 100%|██████████| 339/339 [01:35<00:00,  3.56it/s] 


Top changes:
  Changed: model.layers.0.self_attn.q_proj.weight   Δ=356.6832
  Changed: model.layers.0.self_attn.k_proj.weight   Δ=44.7335
  Changed: model.layers.0.self_attn.v_proj.weight   Δ=45.9836
  Changed: model.layers.0.self_attn.o_proj.weight   Δ=329.3576
  Changed: model.layers.0.mlp.down_proj.weight      Δ=867.7760


In [17]:
if diffs:
    print("\nTop changes:")
    for k, d in diffs[:10]:
        print(f"  Changed: {k:40s} Δ={d:.4f}")
else:
    print("No parameter differences found!")


Top changes:
  Changed: model.layers.0.self_attn.q_proj.weight   Δ=356.6832
  Changed: model.layers.0.self_attn.k_proj.weight   Δ=44.7335
  Changed: model.layers.0.self_attn.v_proj.weight   Δ=45.9836
  Changed: model.layers.0.self_attn.o_proj.weight   Δ=329.3576
  Changed: model.layers.0.mlp.down_proj.weight      Δ=867.7760
  Changed: model.layers.1.self_attn.q_proj.weight   Δ=348.2190
  Changed: model.layers.1.self_attn.k_proj.weight   Δ=56.1504
  Changed: model.layers.1.self_attn.v_proj.weight   Δ=48.3238
  Changed: model.layers.1.self_attn.o_proj.weight   Δ=385.6129
  Changed: model.layers.1.mlp.down_proj.weight      Δ=954.3633
